[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/es/lab3/lab3_parte2.ipynb)
# Práctica 3: Redes neuronales usando PyTorch Lightning
## Parte 2. Usando PyTorch Lightning

Con PyTorch Lightning podremos simplificar la implementación del bucle de entrenamiento. Lightning nos evita escribir código repetitivo para gestionar el manejo de lotes, la validación y otros aspectos que se repiten en los entrenamientos.

# Pre-requisitos

## Instalar paquetes

En esta primera parte necesitaremos `numpy`, `torch` y `lightning` (y `pandas`, `sklearn` y `seaborn` para cargar el conjunto de datos).

In [33]:
import numpy as np
import torch
import seaborn as sns
import pandas as pd
import lightning as L

np.random.seed(1234567)

# La clase LightningModule

Para utilizar Lightning, tenemos que hacer que nuestro módulo herede de `L.LightningModule` en lugar de `nn.Module`. Además, debemos definir los siguientes métodos:
1. `training_step(self, batch, batch_idx)`, que indica cómo computar la pérdida de un lote
1. `configure_optimizers(self)`, que define qué optimizador usar

In [34]:
import torch.nn as nn
import torch.optim as optim

tamano_entrada = 10 # El conjunto que utilizaremos tiene 10 variables
h0_size = 5
h1_size = 3

class OurLightningNetwork(L.LightningModule):
    def __init__(self):
        super().__init__()
        # TODO - Crea una única capa, que sea un Sequential como el descrito en la parte 1
        self.layer = nn.Sequential(
            nn.Linear(tamano_entrada, h0_size),
            nn.Sigmoid(),
            nn.Linear(h0_size, h1_size),
            nn.Sigmoid(),
            nn.Linear(h1_size, 1)
        )

    def forward(self, x):
        # TODO - Completar
        return self.layer(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        # TODO - Completa lo necesario para calcular la pérdida a partir de x e y. Usa la función de pérdida adecuada.
        y_pred = self(x)
        fn_perdida = nn.BCEWithLogitsLoss()
        loss = fn_perdida(y_pred, y)
        # Hacemos que se muestre el valor del pérdida en cada paso del entrenamiento
        self.log("train_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def configure_optimizers(self):
        # TODO - Define un optimizador adecuado
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        return optimizer

# Instanciamos el modelo
model = OurLightningNetwork()

# Verificación: contamos parámetros entrenables
print("Número de tensores de pesos y bias:", len(list(model.parameters())))


Número de tensores de pesos y bias: 6


## Cargamos el conjunto de datos

Repetimos la carga de los laboratorios anteriores.

In [35]:
# TODO - Copia la carga de datos del notebook anterior
from sklearn.preprocessing import StandardScaler
def load_titanic():
    # Cargamos el dataset Titanic desde seaborn
    df = sns.load_dataset('titanic')

    # 1️⃣ Selección de variables relevantes y limpieza
    # Columnas que vamos a usar
    cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    df = df[cols].copy()

    # Eliminamos filas con valores faltantes
    df = df.dropna(subset=['age', 'embarked', 'fare'])

    # 2️⃣ Separar etiquetas y características
    y = df['survived'].to_numpy().astype(np.float32)        # etiquetas como float
    X = df.drop(columns=['survived'])

    # 3️⃣ One-hot encoding para todas las variables categóricas
    categorical_cols = ['pclass', 'sex', 'embarked', 'alone']
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)  # drop_first=True evita multicolinealidad

    # 4️⃣ Variables numéricas
    numeric_cols = ['age', 'sibsp', 'parch', 'fare']
    X_numeric = X_encoded[numeric_cols + [c for c in X_encoded.columns if c not in numeric_cols]]
    scaler = StandardScaler()
    X_numeric[numeric_cols] = scaler.fit_transform(X_numeric[numeric_cols])

    # 5️⃣ Convertir a numpy arrays
    X_np = X_numeric.to_numpy().astype(np.float32)
    y_np = y.reshape(-1, 1).astype(np.float32)  # reshape para que sea (n_samples,1)

    return X_np, y_np

vectores_x, etiquetas = load_titanic()
vectores_x = torch.from_numpy(vectores_x)
etiquetas = torch.from_numpy(etiquetas)


# Lightning necesita los datos en un dataloader, así que los envolvemos con uno
from torch.utils.data import TensorDataset, DataLoader
# Dataset
dataset = TensorDataset(vectores_x, etiquetas)

# Dataloader (mezcla y divide en batches)
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

## Entrenamiento del modelo

En Lightning no tendremos que escribir el bucle de entrenamiento, simplemente conseguiremos un objeto `Trainer` y llamaremos a su método `fit` indicando los datos a los que queremos ajustar el modelo.



In [36]:
# train the model (hint: here are some helpful Trainer arguments for rapid idea iteration)
trainer = L.Trainer(max_epochs=100)
trainer.fit(model=model, train_dataloaders=dataloader)


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities

┏━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layer │ Sequential │     77 │ train │     0 │
└───┴───────┴────────────┴────────┴───────┴───────┘

Trainable params: 77                                                                                               
Non-trainable params: 0                                                                                            
Total params: 77                                                                                                   
Total estimated model params size (MB): 0.000                                                                      
Modules in train mode: 6                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (23) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.
